# L71 · Whisper 解码策略 — 贪婪解码与 beam search 从原理到实现

**学习目标**
- 理解自回归（autoregressive，AR）解码：每步用之前的 token 预测下一个
- 实现贪婪解码（greedy）和简单 beam search（width=2）
- 了解 Whisper 的特殊 token 体系

← **上一课**　[L70 · Whisper 架构解析](L70_whisper_arch.ipynb)

> 上节课学习了 **Whisper 架构解析**：Log-Mel 输入、Transformer Encoder-Decoder、多任务头。  
> 本课将探讨 **Whisper 解码策略**。

## 自回归解码：token-by-token 生成

 Whisper 输出的是 token 概率序列。解码 = 从概率分布中反复采样，直到遇到结束符。

**贪婪解码**：每步选概率最大的 token。快，O(T×V)（每步对全部词汇表取 argmax），但容易陷入重复。

**Beam search（宽度 W）**：保留 W 条最优路径，每步扩展 W×vocab_size 个候选，保留前 W 个。慢（O(T×W×vocab_size)），质量更好。

### Whisper 的特殊 token
```
<|startoftranscript|>  — 解码起始符
<|en|>                 — 语言标签（英语）
<|transcribe|>         — 任务标签
<|notimestamps|>       — 不输出时间戳
<|endoftext|>          — 解码结束符
```
解码时强制在序列开头插入这些特殊 token，引导模型行为。

In [ ]:
import numpy as np

In [ ]:
# 模拟一个玩具 LM：词汇表大小 10，5 步解码
np.random.seed(7)
VOCAB_SIZE = 10
EOT = 9            # end-of-text token id
MAX_STEPS = 8

def fake_lm(context: list) -> np.ndarray:
    """玩具语言模型：基于上文输出 logits。（模拟，真实中是 Transformer 前向传播）"""
    rng = np.random.RandomState(sum(context) % 137)
    logits = rng.randn(VOCAB_SIZE)
    if len(context) >= 4:
        logits[EOT] += 2.0   # 超过 4 步后更容易结束
    return logits

def softmax(logits):
    x = logits - logits.max()
    e = np.exp(x)
    return e / e.sum()

## ✏️ 任务 1：贪婪解码

In [ ]:
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    """贪婪解码：每步选概率最大的 token。

    算法步骤提示：
      1. 初始化 tokens = list(prompt)
      2. 循环 max_steps 次：
         a. logits = fake_lm(tokens)
         b. probs  = softmax(logits)
         c. next_token = int(np.argmax(probs))   # 取概率最大的 token
         d. tokens.append(next_token)
         e. 若 next_token == eot：break
      3. 返回 tokens
    """
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

prompt = [0, 1, 2]   # 模拟 [<|startoftranscript|>, <|en|>, <|transcribe|>]
try:
    result = greedy_decode(prompt)
    print(f'贪婪解码结果: {result}')
    # 验证 1：前缀保持不变
    assert result[0:3] == prompt, '前缀应保持不变'
    # 验证 2：以 EOT 结束或达到最大步数（终止条件）
    assert result[-1] == EOT or len(result) - len(prompt) == MAX_STEPS, (
        f'应以 EOT 结束或达到 MAX_STEPS={MAX_STEPS} 步，实际序列: {result}'
    )
    print('✅ 贪婪解码验证通过')
except NotImplementedError:
    result = None
    print('⚠️  greedy_decode 尚未实现，result 设为 None（占位）')


## ✏️ 任务 2：Beam Search（宽度 W=2）

In [ ]:
def beam_search(
    prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT
) -> list:
    """简单 beam search。返回分数最高的序列。

    分数定义：该路径所有 log P(token_i) 之和（越大越好）。
      score(seq) = Σ log P(seq[i] | seq[:i])  for i in range(len(prompt), len(seq))

    算法步骤提示：
      1. 初始化 beams = [(0.0, list(prompt))]   # (累积 log-prob, token 序列)
      2. 循环 max_steps 次：
         a. candidates = []
         b. 对每个 (score, tokens) in beams：
              logits = fake_lm(tokens)
              probs  = softmax(logits)
              for token_id in range(VOCAB_SIZE):
                  new_score = score + np.log(probs[token_id] + 1e-12)
                  candidates.append((new_score, tokens + [token_id]))
         c. 按 score 降序排列 candidates，取前 width 个 → 新 beams
         d. 若所有 beam 均以 EOT 结尾：break
      3. 返回 beams[0][1]（分数最高的序列）
    """
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

try:
    beam_result = beam_search(prompt, width=2)
    print(f'Beam search 结果 (W=2): {beam_result}')
    if result is not None:
        print(f'贪婪结果:               {result}')
    print('（两者可能不同——beam search 探索更多路径）')

    # 验证 1：返回值以 prompt 为前缀
    assert beam_result[:len(prompt)] == prompt, 'beam_result 应以 prompt 为前缀'

    # 验证 2：beam search log-prob ≥ greedy log-prob（最优性保证）
    if result is not None:
        def compute_logprob(seq):
            score = 0.0
            tokens = list(prompt)
            for tok in seq[len(prompt):]:
                logits = fake_lm(tokens)
                probs = softmax(logits)
                score += np.log(probs[tok] + 1e-12)
                tokens.append(tok)
            return score
        beam_score   = compute_logprob(beam_result)
        greedy_score = compute_logprob(result)
        assert beam_score >= greedy_score - 1e-6, (
            f'beam search 分数 ({beam_score:.4f}) 应 ≥ greedy 分数 ({greedy_score:.4f})'
        )
    print('✅ Beam search 验证通过')
except NotImplementedError:
    print('⚠️  beam_search 尚未实现，跳过验证')


## 🔗 Aurora 连接

本课实现的两个解码算法与 Aurora 代码库的对应关系：

| 本课实现 | Aurora 模块 | 说明 |
|---|---|---|
| `greedy_decode` | `aurora.llm.decoding.greedy_decode` | ASR 推理时的快速解码路径 |
| `beam_search` | `aurora.llm.decoding.beam_search` | 生产级解码，Whisper 默认 beam_size=5 |

`aurora.llm.decoding` 模块被 `aurora.audio.asr.WhisperInference.transcribe()` 调用，
完成从 encoder 输出到最终 token 序列的转换。  
L86 将在此基础上扩展 top-k / top-p 采样（`aurora.llm.top_k_sample`）。


## 本课收束

| 策略 | 复杂度 | 特点 |
|---|---|---|
| 贪婪 | O(T×V) | 快，局部最优 |
| Beam W=2 | O(T×W×V) | 更好，代价是 W 倍计算 |
| Top-k / Top-p | O(T×V) | 引入随机性，避免重复（见 L86）|

Whisper 默认使用 beam_size=5 + length_penalty，生产环境中效果最好。

下一步 L72：在真实数据集上用 LoRA 微调 Whisper。

---

→ **下一课**　[L72 · Whisper 微调](L72_whisper_finetune.ipynb)

> 下节课将学习 **Whisper 微调**：LoRA 低秩注入 vs 全参数，中文 / 方言数据适配实战。